# HumanWild Generator — EE243 Final Project


## Step 1 — Clone repo and install dependencies

In [11]:
import os
os.chdir('/content')
!rm -rf /content/WildHuman
!git clone https://github.com/YongtaoGE/WildHuman /content/WildHuman
os.chdir('/content/WildHuman')
print(os.getcwd())

!pip install -q diffusers transformers accelerate omegaconf einops

Cloning into '/content/WildHuman'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 60 (delta 24), reused 49 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 1.91 MiB | 5.84 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/WildHuman


## Step 2 — Download model weights from HuggingFace


In [12]:
import os
from google.colab import files

SDXL_DIR = '/content/WildHuman/weights/sdxl'
VAE_DIR = '/content/WildHuman/weights/vae'
CONTROLNET_DIR = '/content/WildHuman/weights/controlnet'

os.makedirs(SDXL_DIR, exist_ok=True)
os.makedirs(VAE_DIR, exist_ok=True)
os.makedirs(CONTROLNET_DIR, exist_ok=True)

#!hf auth login
!hf download stabilityai/stable-diffusion-xl-base-1.0 --local-dir {SDXL_DIR}
!hf download madebyollin/sdxl-vae-fp16-fix --local-dir {VAE_DIR}
!hf download geyongtao/HumanWild --local-dir {CONTROLNET_DIR}

print('All weights downloaded.')

Fetching 57 files:   0% 0/57 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 57 files: 100% 57/57 [03:27<00:00,  3.64s/it]
Download complete: 100% 76.9G/76.9G [03:27<00:00, 361MB/s]                ✓ Downloaded
  path: /content/WildHuman/weights/sdxl
Download complete: 100% 76.9G/76.9G [03:27<00:00, 371MB/s]
Fetching 12 files:   0% 0/12 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Still waiting to acquire lock on /content/WildHuman/weights/vae/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 12 files: 100% 12/12 [00:03<00:00,  3.32it/s]
Download complete: 100% 1.34G/1.34G [00:03<00:00, 884MB/s]                ✓ Downloaded
  path: /content/WildHuman/weights/vae
Download complete: 100% 1.34G/1.34G [00:03<00:00, 369MB/s]
Fetching 4 files:   0% 0/4 [00:

## Step 3 — Upload sample normal maps


In [19]:
# Upload normal maps
os.makedirs('data/custom_normals', exist_ok=True)
print('Upload normal map images (PNG):')
uploaded = files.upload()
for fname in uploaded.keys():
    os.rename(fname, f'data/custom_normals/{fname}')
    print(f'Saved: {fname}')

Upload normal map images (PNG):


Saving 0002.png to 0002.png
Saving 0004.png to 0004.png
Saving 0005.png to 0005.png
Saved: 0002.png
Saved: 0004.png
Saved: 0005.png


## Step 4 — Run generation experiments

We test three failure modes from the paper:
1. **Extreme poses** — unusual normal maps (inverted, split legs, contorted)
2. **Mirror flip** — asymmetric poses, check if output is left-right flipped
3. **Prompt vs pose conflict** — give a normal map one pose but a conflicting text prompt

In [14]:
!sed -i 's/from transformers import DPTFeatureExtractor, DPTForDepthEstimation/from transformers import AutoImageProcessor as DPTFeatureExtractor, DPTForDepthEstimation/' /content/WildHuman/demo.py

!sed -i 's/from PIL import Image$/from PIL import Image\nfrom PIL.Image import Resampling/' /content/WildHuman/demo.py

!pip install -q compel

In [17]:
import subprocess, os
os.chdir('/content/WildHuman')

SDXL_DIR = '/content/WildHuman/weights/sdxl'
VAE_DIR = '/content/WildHuman/weights/vae'
CONTROLNET_DIR = '/content/WildHuman/weights/controlnet'

experiments = [
    ('/content/WildHuman/data/custom_normals/Extreme_normal.png',
     'a gymnast doing a backbend on a beam, photo realistic',
     'extreme', 'gymnast_backbend'),

    ('/content/WildHuman/data/custom_normals/Extreme_normal.png',
     'a yoga instructor doing a backbend outdoors, photo realistic',
     'extreme', 'yoga_backbend'),

    ('/content/WildHuman/data/custom_normals/Extreme_normal.png',
     'a contortionist performing on stage, photo realistic',
     'extreme', 'contortionist_stage'),

]

for normal_path, prompt, category, desc in experiments:
    out_dir = f'/content/WildHuman/outputs/{category}/{desc}'
    os.makedirs(out_dir, exist_ok=True)
    print(f'Generating [{category}]: {desc}')
    result = subprocess.run(
        ['python', 'demo.py',
         '--normal_image_path', normal_path,
         '--prompt', prompt,
         '--controlnet_conditioning_scale', '0.75',
         '--output_dir', out_dir,
         '--sdxl_dir', SDXL_DIR,
         '--vae_dir', VAE_DIR,
         '--normal_controlnet_dir', CONTROLNET_DIR,
         '--num_inference_steps', '50',
         '--num_images_per_prompt', '3'],
        capture_output=True, text=True,
        cwd='/content/WildHuman'
    )
    if result.returncode != 0:
        print(f'  ERROR: {result.stderr[-500:]}')
    else:
        print(f'  Done → {out_dir}')

Generating [head]: headstand_yoga
  Done → /content/WildHuman/outputs/head/headstand_yoga
Generating [head]: headstand_gymnast
  Done → /content/WildHuman/outputs/head/headstand_gymnast
Generating [head]: headstand_park
  Done → /content/WildHuman/outputs/head/headstand_park


In [22]:
import subprocess, os
os.chdir('/content/WildHuman')

SDXL_DIR = '/content/WildHuman/weights/sdxl'
VAE_DIR = '/content/WildHuman/weights/vae'
CONTROLNET_DIR = '/content/WildHuman/weights/controlnet'

experiments = [
    # 0005.png — head tilted up, open hands (target: hand detail)
    ('/content/WildHuman/data/custom_normals/0005.png',
     'a surgeon with hands open ready for operation, photo realistic',
     'hand', 'surgeon_hands'),

    ('/content/WildHuman/data/custom_normals/0005.png',
     'a person doing sign language alphabet, photo realistic',
     'hand', 'sign_language'),

    ('/content/WildHuman/data/custom_normals/0005.png',
     'a pianist with fingers spread on keys, photo realistic',
     'hand', 'pianist'),

    # 0004.png — leaning forward crouching (target: extreme pose)
    ('/content/WildHuman/data/custom_normals/0004.png',
     'a sprinter exploding off starting blocks, photo realistic',
     'extreme', 'sprinter'),

    ('/content/WildHuman/data/custom_normals/0004.png',
     'a rugby player charging forward, photo realistic',
     'extreme', 'rugby'),

    ('/content/WildHuman/data/custom_normals/0004.png',
     'a person stumbling forward off balance, photo realistic',
     'extreme', 'stumbling'),

    # 0002.png — one leg raised, arm extended (target: asymmetry/mirror flip)
    ('/content/WildHuman/data/custom_normals/0002.png',
     'a martial artist performing a high kick, photo realistic',
     'asymmetry', 'martial_kick'),

    ('/content/WildHuman/data/custom_normals/0002.png',
     'a ballet dancer mid leap on stage, photo realistic',
     'asymmetry', 'ballet'),

    ('/content/WildHuman/data/custom_normals/0002.png',
     'a football player kicking a ball, photo realistic',
     'asymmetry', 'football_kick'),
]

for normal_path, prompt, category, desc in experiments:
    out_dir = f'/content/WildHuman/outputs/{category}/{desc}'
    os.makedirs(out_dir, exist_ok=True)
    print(f'Generating [{category}]: {desc}')
    result = subprocess.run(
        ['python', 'demo.py',
         '--normal_image_path', normal_path,
         '--prompt', prompt,
         '--controlnet_conditioning_scale', '0.75',
         '--output_dir', out_dir,
         '--sdxl_dir', SDXL_DIR,
         '--vae_dir', VAE_DIR,
         '--normal_controlnet_dir', CONTROLNET_DIR,
         '--num_inference_steps', '50',
         '--num_images_per_prompt', '3'],
        capture_output=True, text=True,
        cwd='/content/WildHuman'
    )
    if result.returncode != 0:
        print(f'  ERROR: {result.stderr[-500:]}')
    else:
        print(f'  Done → {out_dir}')

Generating [hand]: surgeon_hands
  Done → /content/WildHuman/outputs/hand/surgeon_hands
Generating [hand]: sign_language
  Done → /content/WildHuman/outputs/hand/sign_language
Generating [hand]: pianist
  Done → /content/WildHuman/outputs/hand/pianist
Generating [extreme]: sprinter
  Done → /content/WildHuman/outputs/extreme/sprinter
Generating [extreme]: rugby
  Done → /content/WildHuman/outputs/extreme/rugby
Generating [extreme]: stumbling
  Done → /content/WildHuman/outputs/extreme/stumbling
Generating [asymmetry]: martial_kick
  Done → /content/WildHuman/outputs/asymmetry/martial_kick
Generating [asymmetry]: ballet
  Done → /content/WildHuman/outputs/asymmetry/ballet
Generating [asymmetry]: football_kick
  Done → /content/WildHuman/outputs/asymmetry/football_kick


## Download all results

In [24]:
import shutil
from google.colab import files
shutil.make_archive('humanwild_results', 'zip', 'outputs')
files.download('humanwild_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>